# VIII PPGEE Workshop 2026
### PYTHON PARA ENERGIA SOLAR FOTOVOLTAICA: BIBLIOTECAS, BASES DE DADOS E MODELAGEM NA PRÁTICA
1) Importação de Dados
2) Decomposição
3) Transposição
4) Temperatura do Módulo
5) Curva IV
6) Perdas do Sistema
7) Geração AC
8) Performance


In [ ]:
# Importing Libraries
import pvlib
from pvlib.location import Location
from pvlib import irradiance
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime, timedelta
import pytz

In [ ]:
# Mossoró-RN
Latitude = -5.1872
Longitude = -37.3474
Altitude = 18  # Altitude in meters

# Campinas
Latitude = -22.9056
Longitude = -47.0608
Altitude = 650  # Altitude in meters

loc = Location(Latitude, 
               Longitude, 
               'America/Sao_Paulo',
               Altitude)

In [ ]:
pvgis_tmy_data, _ = pvlib.iotools.get_pvgis_tmy(loc.latitude, loc.longitude, outputformat='json', 
                            usehorizon=True, userhorizon=None, 
                            startyear=2006, endyear=2017, 
                            map_variables=True, url='https://re.jrc.ec.europa.eu/api/', 
                            timeout=30, roll_utc_offset=-3, coerce_year=1990)

display(pvgis_tmy_data)

In [ ]:
import pandas as pd

# assuming pvgis_tmy_data.ghi is your Series with a DatetimeIndex
months_selected = (
    pvgis_tmy_data.ghi
    .groupby(pvgis_tmy_data.index.month)
    .apply(lambda x: x.index.year.unique())
)

print(months_selected)

In [ ]:
pvgis_components_hourly_data, _ = pvlib.iotools.get_pvgis_hourly(loc.latitude, loc.longitude, start=2006, end=2017, 
                                                      raddatabase=None, components=True, surface_tilt=0, 
                                                      surface_azimuth=180, outputformat='json', 
                                                      usehorizon=True, userhorizon=None, pvcalculation=False, 
                                                      peakpower=None, pvtechchoice='crystSi', mountingplace='free', 
                                                      loss=0, trackingtype=0, optimal_surface_tilt=False, 
                                                      optimalangles=False, url='https://re.jrc.ec.europa.eu/api/', 
                                                      map_variables=True, timeout=30)
pvgis_components_hourly_data.head(24)

In [ ]:
pvgis_ghi_hourly_data, _ = pvlib.iotools.get_pvgis_hourly(loc.latitude, loc.longitude, start=2006, end=2017, 
                                                      raddatabase=None, components=False, surface_tilt=0, 
                                                      surface_azimuth=180, outputformat='json', 
                                                      usehorizon=True, userhorizon=None, pvcalculation=False, 
                                                      peakpower=None, pvtechchoice='crystSi', mountingplace='free', 
                                                      loss=0, trackingtype=0, optimal_surface_tilt=False, 
                                                      optimalangles=False, url='https://re.jrc.ec.europa.eu/api/', 
                                                      map_variables=True, timeout=180)

pvgis_ghi_hourly_data.head(24)

In [ ]:
'''def matrix_sum(df, variable):
    # Extract Year and Month from the datetime index
    df_copy = df.copy()
    df_copy['Year'] = df_copy.index.year
    df_copy['Month'] = df_copy.index.month
    
    # Group by Year and Month and sum the values
    dados_mensais = df_copy.groupby(['Year', 'Month'])[variable].sum().reset_index()
    
    # Create the pivot table
    matriz = dados_mensais.pivot(index='Year', columns='Month', values=variable)/1000
    
    # Rename columns to month abbreviations
    meses = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
             'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    matriz.columns = meses
    
    # Get only the years 2001-2018 (first 18 years) for interannual statistics
    years_data = matriz.iloc[0:18]  # 2001 to 2018
   
    # Arredondar para 2 casas decimais
    matriz = matriz.round(2)

    
    return matriz

# Apply the function to your dataset
matrix_ghi = matrix_sum(pvgis_ghi_hourly_data, 'poa_global')
matrix_ghi'''

In [ ]:
'''matrix_ghi.plot.box(title='Boxplot of Monthly GHI (kWh/m²) from 2006 to 2017', figsize=(12, 6))'''

In [ ]:
'''(pvgis_tmy_data.ghi.resample('M').sum()/1000).plot(title='Monthly GHI (kWh/m²) from 2006 to 2017', figsize=(12, 6))'''

In [ ]:
from pvlib import irradiance

solpos = loc.get_solarposition(pvgis_tmy_data.index)

out_erbs = irradiance.erbs(pvgis_tmy_data['ghi'], solpos['zenith'], pvgis_tmy_data.index)
out_erbs = out_erbs.rename(columns={'dni': 'dni_erbs', 'dhi': 'dhi_erbs'})

In [ ]:
out_erbs['dhi_erbs']['1990-01-01':'1990-01-07'].plot(label = 'DHI Erbs', figsize=(20,6))
pvgis_tmy_data['dhi']['1990-01-01':'1990-01-07'].plot(label = 'Satellite Data')
plt.legend()
plt.show()

In [ ]:
out_erbs['dni_erbs']['1990-01-01':'1990-01-07'].plot(label = 'DNI Erbs', figsize=(20,6))
pvgis_tmy_data['dni']['1990-01-01':'1990-01-07'].plot(label = 'Satellite Data')
plt.legend()
plt.show()

In [ ]:
pressure = pvlib.atmosphere.alt2pres(altitude = loc.altitude)
pvgis_tmy_data['pressure'] = pressure

# ERBS
out_erbs = irradiance.erbs(pvgis_tmy_data['ghi'], solpos['zenith'], pvgis_tmy_data.index)
out_erbs = out_erbs.rename(columns={'dni': 'dni_erbs', 'dhi': 'dhi_erbs'})

# DISC
out_disc = irradiance.disc(
    pvgis_tmy_data['ghi'], solpos.zenith, pvgis_tmy_data.index, pvgis_tmy_data['pressure']*100)
# use "complete sum" AKA "closure" equations: DHI = GHI - DNI * cos(zenith)
df_disc = irradiance.complete_irradiance(
    solar_zenith=solpos.apparent_zenith, ghi=pvgis_tmy_data['ghi'], dni=out_disc.dni,
    dhi=None)
out_disc = out_disc.rename(columns={'dni': 'dni_disc'})
out_disc['dhi_disc'] = df_disc.dhi

# BOLAND
out_boland = irradiance.boland(pvgis_tmy_data['ghi'], solpos.zenith, pvgis_tmy_data.index)
out_boland = out_boland.rename(
    columns={'dni': 'dni_boland', 'dhi': 'dhi_boland'})

# DIRINT
dni_dirint = irradiance.dirint(
    pvgis_tmy_data['ghi'], solpos.zenith, pvgis_tmy_data.index, pvgis_tmy_data['pressure']*100,
    temp_dew=None)
# use "complete sum" AKA "closure" equation: DHI = GHI - DNI * cos(zenith)
df_dirint = irradiance.complete_irradiance(
    solar_zenith=solpos.apparent_zenith, ghi=pvgis_tmy_data['ghi'], dni=dni_dirint,
    dhi=None)
out_dirint = pd.DataFrame(
    {'dni_dirint': dni_dirint, 'dhi_dirint': df_dirint.dhi},
    index=pvgis_tmy_data.index)

# LOUCHE
out_louche = irradiance.louche(pvgis_tmy_data['ghi'], solpos.zenith, pvgis_tmy_data.index)
out_louche = out_louche.rename(
    columns={'dni': 'dni_louche', 'dhi': 'dhi_louche'})


In [ ]:
#Comparando DHI
out_erbs['dhi_erbs']['1990-01-01':'1990-01-07'].plot(label = 'DHI Erbs', figsize=(20,6))
out_dirint['dhi_dirint']['1990-01-01':'1990-01-07'].plot(label = 'DHI Dirint')
out_disc['dhi_disc']['1990-01-01':'1990-01-07'].plot(label = 'DHI Disc')
out_boland['dhi_boland']['1990-01-01':'1990-01-07'].plot(label = 'DHI Boland')
pvgis_tmy_data['dhi']['1990-01-01':'1990-01-07'].plot(label = 'Satellite Data')
print(out_erbs['dhi_erbs']['1990-01-01':'1990-01-07'].sum())
print(out_dirint['dhi_dirint']['1990-01-01':'1990-01-07'].sum())
print(out_disc['dhi_disc']['1990-01-01':'1990-01-07'].sum())
print(out_boland['dhi_boland']['1990-01-01':'1990-01-07'].sum())
print(pvgis_tmy_data['dhi']['1990-01-01':'1990-01-07'].sum())
plt.legend()
plt.show()

In [ ]:
#Comparando DNI
out_erbs['dni_erbs']['1990-01-01':'1990-01-07'].plot(label = 'DNI Erbs', figsize=(20,6))
out_dirint['dni_dirint']['1990-01-01':'1990-01-07'].plot(label = 'DNI Dirint')
out_disc['dni_disc']['1990-01-01':'1990-01-07'].plot(label = 'DNI Disc')
out_boland['dni_boland']['1990-01-01':'1990-01-07'].plot(label = 'DNI Boland')
pvgis_tmy_data['dni']['1990-01-01':'1990-01-07'].plot(label = 'Satellite Data')
print(out_erbs['dni_erbs']['1990-01-01':'1990-01-07'].sum())
print(out_dirint['dni_dirint']['1990-01-01':'1990-01-07'].sum())
print(out_disc['dni_disc']['1990-01-01':'1990-01-07'].sum())
print(out_boland['dni_boland']['1990-01-01':'1990-01-07'].sum())
print(pvgis_tmy_data['dni']['1990-01-01':'1990-01-07'].sum())
plt.legend()
plt.show()

In [ ]:
# Transposição de Irradiância

surface_tilt = 28
surface_azimuth = 0
solar_zenith = solpos['zenith']
solar_azimuth = solpos['azimuth']
dni = pvgis_tmy_data['dni']
ghi = pvgis_tmy_data['ghi']
dhi = pvgis_tmy_data['dhi']
dni_extra = pvlib.irradiance.get_extra_radiation(pvgis_tmy_data.index)
airmass_relative = pvlib.atmosphere.get_relative_airmass(solpos['apparent_zenith'])
pressure = pvlib.atmosphere.alt2pres(altitude = loc.altitude)
airmass = pvlib.atmosphere.get_absolute_airmass(airmass_relative, pressure)

POA_Irradiance = pvlib.irradiance.get_total_irradiance(surface_tilt,
                                                  surface_azimuth,
                                                  solar_zenith,
                                                  solar_azimuth,
                                                  dni,
                                                  ghi,
                                                  dhi,
                                                  dni_extra=dni_extra,
                                                  airmass=airmass,
                                                  albedo=0.25,
                                                  surface_type=None,
                                                  model='perez', #'isotropic', 'klucher', 'haydavies', 'reindl', 'king', 'perez', 'perez-driesse'
                                                  model_perez='allsitescomposite1990') #Used only for Perez Model)
# Dados
poa = POA_Irradiance['poa_global']
ghi = pvgis_tmy_data['ghi']

# 1. Plot comparativo temporal aprimorado
plt.figure(figsize=(18, 8))
ax = plt.gca()

# Plot das séries com preenchimento entre elas
poa.plot(label='POA Global (Plano do Arranjo)', color='#FF7F0E', linewidth=1.8, ax=ax)
ghi.plot(label='GHI (Global Horizontal)', color='#1F77B4', linewidth=1.8, ax=ax)
plt.fill_between(poa.index, poa, ghi, where=(poa >= ghi), 
                facecolor='orange', alpha=0.3, interpolate=True)
plt.fill_between(poa.index, poa, ghi, where=(poa < ghi), 
                facecolor='blue', alpha=0.3, interpolate=True)

# Configurações do gráfico
plt.title('Comparação entre Irradiância no Plano do Arranjo (POA) e Global Horizontal (GHI)', 
          fontsize=14, pad=20)
plt.xlabel('Data', fontsize=12)
plt.ylabel('Irradiância (W/m²)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)

# Adicionando anotações de valores máximos
max_poa = poa.max()
max_ghi = ghi.max()
plt.annotate(f'Max POA: {max_poa:.1f} W/m²', 
             xy=(poa.idxmax(), max_poa), xytext=(10, 10),
             textcoords='offset points', ha='left', va='bottom',
             bbox=dict(boxstyle='round,pad=0.5', fc='orange', alpha=0.5),
             arrowprops=dict(arrowstyle='->'))
plt.annotate(f'Max GHI: {max_ghi:.1f} W/m²', 
             xy=(ghi.idxmax(), max_ghi), xytext=(10, -25),
             textcoords='offset points', ha='left', va='top',
             bbox=dict(boxstyle='round,pad=0.5', fc='lightblue', alpha=0.5),
             arrowprops=dict(arrowstyle='->'))

plt.tight_layout()
plt.show()

# 2. Cálculo de métricas básicas
metrics = {
    'Máximo (W/m²)': [poa.max(), ghi.max()],
    'Média (W/m²)': [poa.mean(), ghi.mean()],
    'Energia Total (kWh/m²)': [poa.sum()/1000, ghi.sum()/1000]
}

metrics_df = pd.DataFrame(metrics, index=['POA', 'GHI'])

# 3. Exibição das métricas formatadas
print("\n=== Métricas Comparativas ===")
print(metrics_df.to_string(float_format=lambda x: f"{x:.2f}"))

# 4. Análise da razão diária POA/GHI (se houver dados suficientes)
if len(poa) > 24:  # Pelo menos 1 dia de dados horários
    daily_poa = poa.resample('D').sum()/1000
    daily_ghi = ghi.resample('D').sum()/1000
    daily_ratio = daily_poa/daily_ghi
    
    plt.figure(figsize=(15, 6))
    daily_ratio.plot(color='green', linewidth=1.5)
    plt.axhline(y=1, color='r', linestyle='--', label='Razão = 1')
    plt.title('Variação Diária da Razão POA/GHI', fontsize=14)
    plt.ylabel('Razão POA/GHI', fontsize=12)
    plt.xlabel('Data', fontsize=12)
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("\nDados insuficientes para análise diária (mínimo 24 horas requeridas)")

In [ ]:
'''# Ângulo Ótimo do Módulo
import numpy as np
import pandas as pd
import pvlib
import matplotlib.pyplot as plt

# Faixa de inclinações a testar (ex.: 0° a 60°)
tilts = np.arange(0, 91, 1)

results = []

# Variáveis fixas
surface_azimuth = 0  # módulo orientado para o Norte (sul seria 180)
solar_zenith = solpos['zenith']
solar_azimuth = solpos['azimuth']

dni = pvgis_tmy_data['dni']
ghi = pvgis_tmy_data['ghi']
dhi = pvgis_tmy_data['dhi']

dni_extra = pvlib.irradiance.get_extra_radiation(pvgis_tmy_data.index)
airmass_relative = pvlib.atmosphere.get_relative_airmass(solpos['apparent_zenith'])
pressure = pvlib.atmosphere.alt2pres(loc.altitude)
airmass = pvlib.atmosphere.get_absolute_airmass(airmass_relative, pressure)


for tilt in tilts:
    
    # Ângulo de incidência
    aoi = pvlib.irradiance.aoi(
        tilt,
        surface_azimuth,
        solar_zenith,
        solar_azimuth
    )

    # IAM usando modelo Ashrae
    iam = pvlib.iam.ashrae(aoi, b=0.05)

    # Transposição (POA)
    POA_Irradiance = pvlib.irradiance.get_total_irradiance(
        tilt,
        surface_azimuth,
        solar_zenith,
        solar_azimuth,
        dni,
        ghi,
        dhi,
        dni_extra=dni_extra,
        airmass=airmass,
        albedo=0.25,
        model='perez',
        model_perez='allsitescomposite1990'
    )

    poa = POA_Irradiance['poa_direct'] * iam + POA_Irradiance['poa_diffuse']
    poa = poa.fillna(0)

    # Ganho de transposição anual
    gain = poa.sum() / ghi.sum()

    results.append([tilt, gain])


# DataFrame com resultados
df_gain = pd.DataFrame(results, columns=['tilt', 'gain'])

# Encontrar tilt ótimo
best_row = df_gain.loc[df_gain['gain'].idxmax()]
best_tilt = best_row['tilt']
best_gain = best_row['gain']

print("=====================================")
print(f"Melhor tilt (surface_tilt): {best_tilt:.1f}°")
print(f"Ganho máximo (POA/GHI): {best_gain:.4f}")
print("=====================================")

# Plot do ganho
plt.figure(figsize=(12,5))
plt.plot(df_gain['tilt'], df_gain['gain'])
plt.scatter(best_tilt, best_gain, color='red', label='Melhor Tilt')
plt.xlabel("Inclinação (°)")
plt.ylabel("Ganho de Transposição (POA/GHI)")
plt.title("Busca do Melhor Ângulo de Inclinação")
plt.grid(True)
plt.show()'''

#### 6 - Modelagem de Temperatura do Módulo FV

In [ ]:
#Características do Módulo
Vmp = 30.8 #Voltage at Maximum Power
Imp = 8.75 #Current at Maximum Power
Voc = 37.9 #Open-Circuit Voltage
Isc = 9.32 #Short-Circuit Current
alpha = 0.05*Isc/100 # Alpha_isc (%/A) * Short-Circuit Current
beta = (-0.31)*Voc/100 # Beta_voc (%/V) * Open-Circuit Voltage
gamma = -0.41
Ns = 60 #Number of PV Cells in Series
A_c = 1.6368 #The Area of PV Module's Surface
module_efficiency = (Vmp*Imp)/(A_c*1000)  # Convertendo para decimal

In [ ]:
#Influência de Parâmetros para a Modelagem de Temperatura
#Pedir para plotarem os gráficos com
import matplotlib.pyplot as plt

# Criar figura com 4 subplots
fig, axes = plt.subplots(nrows=4, ncols=1, figsize=(20, 20))

# Ajustar espaçamento entre subplots
plt.subplots_adjust(hspace=0.4)

# Gráfico 1: poa.plot()
poa['1990-01-01':'1990-01-07'].plot(ax=axes[0], title='POA (Plane of Array) Irradiance', color='orange')
axes[0].set_ylabel('Irradiance (W/m²)')
axes[0].grid(True)

# Gráfico 2: Temperatura do ar
pvgis_tmy_data['temp_air']['1990-01-01':'1990-01-07'].plot(ax=axes[1], title='Air Temperature', color='red')
axes[1].set_ylabel('Temperature (°C)')
axes[1].grid(True)

# Gráfico 3: Velocidade do vento
pvgis_tmy_data['wind_speed']['1990-01-01':'1990-01-07'].plot(ax=axes[2], title='Wind Speed', color='blue')
axes[2].set_ylabel('Speed (m/s)')
axes[2].grid(True)

# Gráfico 4: Temperatura da célula PV
PV_Cell_Temperature_PVsyst = pvlib.temperature.pvsyst_cell(poa_global = poa, 
                                                           temp_air = pvgis_tmy_data['temp_air'], 
                                                           wind_speed = pvgis_tmy_data['wind_speed'], 
                                                           u_c = 29.0, u_v=0.0, 
                                                           module_efficiency=module_efficiency, 
                                                           alpha_absorption=0.9)
PV_Cell_Temperature_PVsyst['1990-01-01':'1990-01-07'].plot(title='Temperatura da Célula Fotovoltaica - Modelo PVsyst')
axes[3].set_ylabel('Temperature (°C)')
axes[3].grid(True)

# Ajustar layout
plt.tight_layout()
plt.show()

#### 7 - Single Diode Model para Geração FV

In [ ]:
#Bibliotecas
from pvlib import pvsystem
from scipy.constants import Boltzmann, elementary_charge

In [ ]:
def desotorefparameters(Vmp, Imp, Voc, Isc, alpha, beta, Ns, EgRef, dEgdT, Tref, Gref):
    desoto_fit_params = pvlib.ivtools.sdm.fit_desoto(Vmp,
                                                     Imp,
                                                     Voc, 
                                                     Isc,
                                                     alpha,
                                                     beta,
                                                     Ns,
                                                     EgRef,
                                                     dEgdT,
                                                     Tref, 
                                                     Gref,
                                                     root_kwargs = {'method': 'lm', 'options':{'maxiter':10000, 'xtol': 1e-3, 'gtol': 1e-3}})
    desoto_fit_params[0]

    fitted_params = desoto_fit_params[0]

    desoto_params_dict = fitted_params

    results_df_desoto = pd.DataFrame(columns = ['a_ref', 'Ipv_ref', 'Io_ref', 'Rp_ref', 'R_s'])

    results_df_desoto.loc[1] = [desoto_params_dict['a_ref'], desoto_params_dict['I_L_ref'], desoto_params_dict['I_o_ref'],
                                desoto_params_dict['R_sh_ref'], desoto_params_dict['R_s']]
    return(results_df_desoto)

def desotoparametersopc(irrad, temp_cell, alpha, a_ref, Ipv_ref, Io_ref, Rp_ref, R_s, EgRef, dEgdT, Gref, Tref, method = 'lambertw', number_of_points = 100):
    diode_params_desoto = pvlib.pvsystem.calcparams_desoto(irrad, 
                                                          temp_cell, 
                                                          alpha, 
                                                          a_ref, 
                                                          Ipv_ref, 
                                                          Io_ref, 
                                                          Rp_ref, 
                                                          R_s, 
                                                          EgRef, 
                                                          dEgdT, 
                                                          Gref, 
                                                          Tref)
    SDE_params = {
        'photocurrent': diode_params_desoto[0],
        'saturation_current': diode_params_desoto[1],
        'resistance_series': diode_params_desoto[2],
        'resistance_shunt': diode_params_desoto[3],
        'nNsVth': diode_params_desoto[4]
    }

    curve_info = pvsystem.singlediode(method=method, **SDE_params)
    v = pd.DataFrame(np.linspace(0, curve_info['v_oc'], number_of_points))
    i = pd.DataFrame(pvsystem.i_from_v(voltage=v, method=method, **SDE_params))

    return ({'Key_Points': curve_info,
             'Voltage': v,
             'Current': i})

#Constantes necessárias para a modelagem em STC
EgRef =1.121 #Valence energy band-gap for Crystalline Silicon
dEgdT = - 0.0002677 #
Tref  = 25 #Temperature (°C) under STC
Gref = 1000 #Irradiance (W/m²) under STC
k = Boltzmann
q = elementary_charge

In [ ]:
desoto_ref_parameters = desotorefparameters(Vmp, Imp, Voc, Isc, alpha, beta, Ns, EgRef, dEgdT, Tref, Gref)
print('Os 5 parâmetros do SDM em STC')
desoto_ref_parameters

In [ ]:
a_ref = desoto_ref_parameters['a_ref'][1]
Ipv_ref = desoto_ref_parameters['Ipv_ref'][1]
Io_ref = desoto_ref_parameters['Io_ref'][1]
Rp_ref = desoto_ref_parameters['Rp_ref'][1]
R_s = desoto_ref_parameters['R_s'][1]

irrad = poa
temp_cell = PV_Cell_Temperature_PVsyst

results = desotoparametersopc(irrad, temp_cell, alpha, a_ref, Ipv_ref, Io_ref, Rp_ref, R_s, EgRef, dEgdT, Gref, Tref, method = 'lambertw', number_of_points = 100)

Nm = 10

PV_Gen = results['Key_Points']['p_mp']*Nm
PV_Gen.plot(figsize=(20,6))

#### 8 - Perdas do Sistema e do Inversor FV

In [ ]:
losses = pvlib.pvsystem.pvwatts_losses(soiling=2, 
                                       shading=3, 
                                       snow=0, 
                                       mismatch=2, 
                                       wiring=2, 
                                       connections=0.5, 
                                       lid=1.5, 
                                       nameplate_rating=1, 
                                       age=0, 
                                       availability=3)

PV_Array_Gen = PV_Gen * (100 - losses)/100
PV_Array_Gen.resample('ME').sum().plot(figsize=(20,6), label = 'Com perdas', title='Potência Gerada pelo Sistema Fotovoltaico')
PV_Gen.resample('ME').sum().plot(figsize=(20,6), label = 'Sem perdas', title='Potência Gerada pelo Sistema Fotovoltaico')
plt.legend()

In [ ]:
PV_AC = pvlib.inverter.pvwatts(pdc = PV_Array_Gen, 
                               pdc0 = Vmp*Imp*Nm, 
                               eta_inv_nom=0.96, 
                               eta_inv_ref=0.9637)

PV_Gen.resample('D').sum().plot(figsize=(20,6), label = 'Sem perdas', title='Potência Gerada pelo Sistema Fotovoltaico')
PV_Array_Gen.resample('D').sum().plot(figsize=(20,6), label = 'Com perdas', title='Potência DC Gerada pelo Arranjo Fotovoltaico')
PV_AC.resample('D').sum().plot(figsize=(20,6), label = 'AC', title='Potência AC Gerada pelo Inversor Fotovoltaico')
plt.legend()

In [ ]:
PV_AC_Final = PV_AC.clip(upper=1500)
PV_AC['2024-01-05':'2024-01-05'].plot(label='Sem clipping', figsize=(20,6))
PV_AC_Final['2024-01-05':'2024-01-05'].plot(label='Com clipping', figsize=(20,6))
plt.legend()

In [ ]:
PV_Gen.resample('ME').sum().plot(figsize=(20,6), label = 'Sem perdas', title='Potência Gerada pelo Sistema Fotovoltaico')
PV_Array_Gen.resample('ME').sum().plot(figsize=(20,6), label = 'Com perdas', title='Potência DC Gerada pelo Arranjo Fotovoltaico')
PV_AC.resample('ME').sum().plot(figsize=(20,6), label = 'AC', title='Potência AC Gerada pelo Inversor Fotovoltaico')
PV_AC_Final.resample('ME').sum().plot(figsize=(20,6), label = 'AC Final', title='Potência AC Final Gerada pelo Inversor Fotovoltaico')
plt.legend()

#### 9 - Métricas de Performance da Usina
1) Yield Final (Yf)
2) Horas de Sol Pleno (HSP)
3) Performance Ratio (PR)
4) Fator de Capacidade (FC)

In [ ]:
# Yf
Yf = PV_AC_Final.resample('ME').sum() / (Vmp*Imp*Nm)
print("=====================================")
print(f"Yield Factor (Yf): {Yf} kWh/kW")

In [ ]:
# HSP
HSP = poa.resample('ME').sum() / 1000
print(f"Hourly Solar Production (HSP): {HSP} h")

In [ ]:
# HSP
HSP = poa.resample('ME').sum() / 1000
print(f"Hourly Solar Production (HSP): {HSP} h")

In [ ]:
PR = Yf / HSP
print(f"Performance Ratio (PR): {PR}")

In [ ]:
PR.plot(figsize=(20,6), title='Performance Ratio (PR) Mensal')

In [ ]:
dias = PV_AC_Final.resample('D').sum().index.size
print(f"Número de dias considerados: {dias} dias")

In [ ]:
FC = PV_AC_Final.resample('YE').sum() / (dias * Vmp * Imp * Nm * 24)
print(f"Capacity Factor (FC): {FC}")

In [ ]:
# Energia mensal (Wh ou kWh, consistente com PV_AC_Final)
energia_mensal = PV_AC_Final.resample('ME').sum()

# Número de dias de cada mês
dias_mes = energia_mensal.index.days_in_month

# Capacity Factor mensal
FC_mensal = energia_mensal / (dias_mes * Vmp * Imp * Nm * 24)

print("Capacity Factor mensal:")
print(FC_mensal)


In [ ]:
FC_mensal.plot(figsize=(20,6), marker='o')
plt.ylabel('Capacity Factor')
plt.grid(True)
plt.show()